# Adaptive Query Execution (AQE): Runtime Query Optimization

**Module**: 07_spark_sql_internals  
**Notebook**: 05_adaptive_query_execution.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Adaptive Query Execution re-optimizes query plans at runtime based on materialized stage statistics: coalescing shuffle partitions, switching join strategies, and optimizing skew joins.
- **Why does this topic exist?**: AQE eliminates static partition guessing and dynamically fixes data skew.
- **What problem does it solve?**: Static queries planned with outdated statistics before knowing actual filtered row counts.
- **Where is it used in real-world Data Engineering?**: Production performance tuning in Spark 3.x and 4.x.

---

## 2. The 3 Pillars of AQE
1. **Dynamically Coalescing Shuffle Partitions**: Merges hundreds of tiny post-shuffle partitions into fewer optimal-sized partitions.
2. **Dynamically Switching Join Strategies**: Converts Sort-Merge Join to Broadcast Hash Join if the filtered build-side table turns out to be smaller than the broadcast threshold.
3. **Dynamically Handling Data Skew**: Automatically detects skewed partitions and splits them into smaller sub-partitions.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("AQE_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("=== Active AQE Configurations in Cluster ===")
print("spark.sql.adaptive.enabled:                    ", spark.conf.get("spark.sql.adaptive.enabled", "true"))
print("spark.sql.adaptive.coalescePartitions.enabled: ", spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled", "true"))
print("spark.sql.adaptive.skewJoin.enabled:           ", spark.conf.get("spark.sql.adaptive.skewJoin.enabled", "true"))

---

## Experiment: Observing AdaptiveSparkPlan in Physical Plan

- **1. What are we testing?**: Verify that query plans are wrapped in AdaptiveSparkPlan.
- **2. What do we expect?**: AdaptiveSparkPlan operator appears at root of plan.

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
res = orders.groupBy("channel").count()
print("Physical plan with AQE:")
res.explain()

**3. What happened?**: AdaptiveSparkPlan operator is present.  
**4. Why did it happen?**: AQE is enabled in cluster defaults.  
**5. Practical Engineering Lesson**: AdaptiveSparkPlan re-evaluates physical operators at each shuffle stage boundary.

---

## Practical Exercises

### Exercise 1 (Beginner): Check AQE Enabled
**Problem Statement**: Verify spark.sql.adaptive.enabled.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Inspect Post-Shuffle Partition Coalescing
**Problem Statement**: Observe how AQE coalesced shuffle partitions in Spark UI SQL tab.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): AQE Skew Join Parameters
**Problem Statement**: Check skewedPartitionFactor setting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check AQE Enabled):
```python
print(spark.conf.get('spark.sql.adaptive.enabled'))
```

#### Solution 2 (Inspect Post-Shuffle Partition Coalescing):
```python
print('Inspected AQE nodes in localhost:4040/SQL')
```

#### Solution 3 (AQE Skew Join Parameters):
```python
print(spark.conf.get('spark.sql.adaptive.skewJoin.skewedPartitionFactor','5'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **AQE Revolution**: Re-optimizes plans at runtime using actual stage metrics.
- **Features**: Coalesce partitions, switch to broadcast join, split skewed partitions.

### Essential Technical Terms
- `AdaptiveSparkPlan`: Runtime adaptive query coordinator.
- `SkewJoin`: Splitting skewed partition chunks.

### Common Engineering Mistakes
- **Disabling AQE**: Forces reliance on static upfront estimates.

### Interview Questions on This Topic
- **Q: What are the three main features of Adaptive Query Execution?**
  - *A*: 1. Dynamically coalescing shuffle partitions, 2. Dynamically converting SortMergeJoin to BroadcastHashJoin, and 3. Dynamically splitting skewed join partitions.

### What to Study Next
Module 07 complete! Proceed to Module 08: Data Sources and File Formats.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')